In [1]:
import pandas as pd
import pickle

from geoai.utils_geo.RasterOps import RasterOperations
from geoai.utils_ds.DataFrameOps import DataFrameOperations
from geoai.utils_geo.VectorOps import VectorOperations

raster_ops = RasterOperations()
df_ops = DataFrameOperations()

In [2]:
RASTER_PATH = r"raster_files\QC_2018_2023.tif"
N_BANDS = 5
BAND_NAMES = ["BLUE", "GREEN", "RED", "NIR", "SWIR"]

In [4]:
df_bands = []
array = raster_ops.raster_to_array(RASTER_PATH)
raster_dimension = raster_ops.get_raster_dimensions(RASTER_PATH)


for band_index, band_name in zip(range(N_BANDS), BAND_NAMES):
    flat = raster_ops.flatten_array(array, band_index)
    df = df_ops.convert_to_df(flat, band_name)
    df = df.loc[~(df == 0).all(axis=1)]  # remove rows if all of its column is zero
    df_bands.append(df)
final_df_per_bands = pd.concat(df_bands, axis=1)


Converting raster array to DataFrame with column name BLUE
Converting raster array to DataFrame with column name GREEN
Converting raster array to DataFrame with column name RED
Converting raster array to DataFrame with column name NIR
Converting raster array to DataFrame with column name SWIR


In [5]:
final_df_per_bands

,BLUE,GREEN,RED,NIR,SWIR
0,1337.500000,1697.000000,1712.000000,2578.000000,2172.5
1,1374.666626,1663.333374,1638.000000,2679.366699,2185.5
2,1370.500000,1576.000000,1592.800049,2183.333252,2202.5
3,1318.000000,1446.666626,1404.000000,1894.833374,2202.5
4,994.333313,1136.666626,1119.000000,1732.000000,1922.0
...,...,...,...,...,...
158219,1012.000000,971.333313,873.000000,2283.333252,2902.5
158220,1049.500000,1049.333374,961.000000,2343.000000,2677.0
158221,1046.000000,1049.333374,975.000000,2310.000000,2677.0
158222,765.333313,771.000000,722.000000,1962.500000,2168.5


In [6]:
final_df_per_bands["NDVI"] = (final_df_per_bands["NIR"] - final_df_per_bands["RED"]) / (final_df_per_bands["NIR"] + final_df_per_bands["RED"])
final_df_per_bands["NDBI"] = (final_df_per_bands["SWIR"] - final_df_per_bands["NIR"]) / (final_df_per_bands["SWIR"] + final_df_per_bands["NIR"])
final_df_per_bands["REI"] = (final_df_per_bands["NIR"] - final_df_per_bands["BLUE"]) / (final_df_per_bands["NIR"] + final_df_per_bands["BLUE"] * final_df_per_bands["NIR"])
final_df_per_bands.fillna(0, inplace=True)

In [ ]:
numerical_columns = final_df_per_bands.select_dtypes(include=['float64']).columns.tolist()
print(numerical_columns)
final_df_per_bands, X_test = preprocess_ops.polynomial_transform(final_df_per_bands, X_test, numerical_columns, 2)
final_df_per_bands.head(1)

In [5]:
raster_ops.column_to_raster(
    "raster_files/PREDICTED_LC.tif",
    final_df_per_bands,
    "PREDICTED_LC",
    raster_dimension,
    "float32",
)

Raster written to raster_files/PREDICTED_LC.tif
